
# 📄 AI-Powered PDF Question Answering System using RAG (Retrieval Augmented Generation)

### Applied Generative and Agentic AI Systems — Course Project

---

## 1. Introduction

### What problem are we solving?

Large Language Models (LLMs) like Google's **Gemini** are extremely good at reasoning and writing
fluent text, but they have two big limitations:

1. **They don't know about *your* documents.** Gemini was trained on public internet data up to a
   certain date — it has never seen the PDF report sitting on your laptop.
2. **They can hallucinate.** If you ask an LLM a question about a document it has never seen, it
   may confidently make up an answer instead of saying "I don't know."

**Retrieval-Augmented Generation (RAG)** solves this by combining two systems:

- A **retriever**, which searches your own documents for the most relevant pieces of text.
- A **generator** (the LLM), which reads only those relevant pieces and writes a grounded answer.

Instead of asking the LLM "What does this 200-page PDF say about X?" (which would never fit in its
context window, and which it hasn't memorized), we ask it: **"Here are the 4 most relevant
paragraphs from the PDF — answer the question using only these."** This is faster, cheaper, more
accurate, and — crucially — it lets us **show the user exactly which text the answer came from**.

### High-level architecture

```
   PDF files
      │
      ▼
 [1] Extract text (PyPDF)
      │
      ▼
 [2] Split into small overlapping chunks (LangChain text splitter)
      │
      ▼
 [3] Convert each chunk into a vector embedding (SentenceTransformers)
      │
      ▼
 [4] Store all vectors in a FAISS vector database (fast similarity search)
      │
      ▼
 [5] User asks a question ──► embed the question ──► FAISS retrieves top-k similar chunks
      │
      ▼
 [6] Retrieved chunks + question are sent to Gemini as a prompt
      │
      ▼
 [7] Gemini generates an answer grounded in the retrieved text
      │
      ▼
 [8] Answer + source chunks + chat history shown in a Gradio chat UI
```

### Tech stack used in this notebook

| Component            | Tool                                   | Purpose                                   |
|-----------------------|-----------------------------------------|--------------------------------------------|
| PDF reading            | `pypdf`                                | Extract raw text from PDF files            |
| Chunking                | `langchain` `RecursiveCharacterTextSplitter` | Split long text into manageable pieces |
| Embeddings               | `sentence-transformers` (`all-MiniLM-L6-v2`) | Convert text into numeric vectors     |
| Vector database           | `FAISS` (via `langchain-community`)  | Store & search embeddings efficiently      |
| LLM (answer generation)    | Google **Gemini API** (`gemini-1.5-flash`) | Generate the final natural-language answer |
| Orchestration                | `LangChain`                        | Glue all the above together                |
| User interface                 | `Gradio`                        | Chat UI that runs directly inside the notebook |

### What you will learn by running this notebook

- How to build a full RAG pipeline from scratch, not just call a pre-built library function.
- How embeddings and vector similarity search actually work.
- How to design prompts that force an LLM to stay "grounded" in retrieved context.
- How to build a simple agentic/chat interface with memory, directly inside Jupyter.

> **Note:** This notebook is designed to run top-to-bottom in order. Run every cell once, in
> sequence, the first time. After that, you can re-run individual cells (e.g. to upload new PDFs)
> without restarting from Section 1.



## 2. Install Required Libraries

We install everything needed for the whole pipeline in one cell:

- `langchain`, `langchain-community`, `langchain-google-genai` — orchestration + Gemini integration
- `pypdf` — reads text out of PDF files
- `sentence-transformers` — generates embeddings locally (free, no API needed for this step)
- `faiss-cpu` — fast vector similarity search database
- `gradio` — the chat UI
- `python-dotenv` — lets us load the Gemini API key from a `.env` file instead of hardcoding it

> **Tip:** In Google Colab / Jupyter, `!pip install` runs a shell command from inside a notebook
> cell. The `-q` flag keeps the output quiet. If you're running this locally, you only need to run
> this cell once per environment.


In [1]:

# Install all required Python packages.
# -q  = quiet mode (less clutter in the output)
# -U  = upgrade to latest compatible version
!pip install -q -U langchain langchain-community langchain-core langchain-text-splitters \
    langchain-huggingface langchain-google-genai \
    pypdf sentence-transformers faiss-cpu gradio python-dotenv



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip



## 3. Imports

Here we import every library we installed above, plus a few standard Python modules.

We also do a **basic sanity check**: printing library versions helps you (and your professor)
confirm the environment is set up correctly, and helps with debugging if something breaks later.


In [1]:

# ---- Standard library imports ----
import os                # For reading environment variables (like our API key)
import getpass            # For securely prompting for the API key if not already set
import traceback           # For readable error messages in our error-handling blocks
from typing import List, Tuple  # Type hints make the code easier to read and debug

# ---- PDF reading ----
from pypdf import PdfReader

# ---- LangChain: text splitting (standalone package in current LangChain versions) ----
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---- LangChain: embeddings (wraps sentence-transformers; standalone integration package) ----
from langchain_huggingface import HuggingFaceEmbeddings

# ---- LangChain: vector store (FAISS integration currently lives in langchain-community) ----
from langchain_community.vectorstores import FAISS

# ---- LangChain: Gemini chat model wrapper ----
from langchain_google_genai import ChatGoogleGenerativeAI

# ---- LangChain: document schema (used to attach metadata like filename/page number to chunks) ----
# NOTE: in current LangChain versions, Document has moved to langchain_core.documents
# (the old `langchain.docstore.document` path is deprecated/removed).
from langchain_core.documents import Document

# ---- Gradio for the UI ----
import gradio as gr

# Print versions so we can confirm everything imported correctly.
import langchain, faiss, sentence_transformers
print("LangChain version:", langchain.__version__)
print("FAISS module loaded successfully:", faiss.__file__ is not None)
print("SentenceTransformers version:", sentence_transformers.__version__)
print("Gradio version:", gr.__version__)
print("\\nAll imports successful ✅")
print("(A DeprecationWarning about 'langchain-community' may appear above — this is expected;")
print(" the FAISS integration still lives in that package as of this writing and works correctly.)")


C:\Users\manik\AppData\Local\Temp\ipykernel_9272\664555783.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


LangChain version: 1.3.14
FAISS module loaded successfully: True
SentenceTransformers version: 5.7.0
Gradio version: 6.22.0
\nAll imports successful ✅
(A DeprecationWarning about 'langchain-community' may appear above — this is expected;
 the FAISS integration still lives in that package as of this writing and works correctly.)



## 4. Load Gemini API Key

### Why do we need an API key?

Google's Gemini models are accessed over the internet through an API. To use them, you need a free
**Gemini API key** from [Google AI Studio](https://aistudio.google.com/app/apikey).

### Best practice: never hardcode your API key in the notebook

If you type your key directly into a code cell like `API_KEY = "AIza..."`, and then upload this
notebook to GitHub, **your key becomes public** and anyone can use (and abuse) it on your account.

Instead, we use this priority order:

1. Check if the key already exists as an environment variable `GOOGLE_API_KEY` (e.g., set by Colab
   secrets, a `.env` file, or your OS).
2. If not found, securely **prompt** for it using `getpass` (input is hidden, like a password
   field, and is never printed or saved to the notebook file).

This way, the key is only ever held in memory for this session, and the notebook itself stays
safe to share.


In [3]:

def get_gemini_api_key() -> str:
    '''
    Returns a valid Gemini API key, obtained either from an environment variable
    or from a secure interactive prompt. Raises a clear error if the user
    cancels/leaves it blank.
    '''
    # Step 1: try to read it from the environment (works well with .env files / Colab secrets)
    api_key = os.environ.get("GOOGLE_API_KEY")

    # Step 2: if not found, ask the user to paste it in securely (input is masked)
    if not api_key:
        print("GOOGLE_API_KEY not found in environment variables.")
        print("Get a free key here: https://aistudio.google.com/app/apikey")
        api_key = getpass.getpass("Paste your Gemini API key here (input is hidden): ").strip()

    # Step 3: basic validation — fail fast with a clear message rather than a cryptic error later
    if not api_key:
        raise ValueError(
            "No Gemini API key was provided. The notebook cannot call the LLM without one."
        )

    # Store it back into the environment so other libraries (e.g. langchain-google-genai)
    # can automatically pick it up without us passing it around everywhere.
    os.environ["GOOGLE_API_KEY"] = api_key
    return api_key


# Wrap the call in a try/except so a bad key or cancelled prompt gives a clean error message
try:
    GEMINI_API_KEY = get_gemini_api_key()
    print("✅ Gemini API key loaded successfully (hidden for security).")
except Exception as e:
    print("❌ Failed to load Gemini API key:", e)
    raise


✅ Gemini API key loaded successfully (hidden for security).



## 5. Read PDFs

### Theory: how do we get text out of a PDF?

A PDF is a *layout* format, not a plain-text format — it stores instructions like "draw this glyph
at position (x, y)" rather than paragraphs of text. Libraries like `pypdf` reconstruct readable text
by reading these instructions page by page.

We will:

1. Accept a **list of PDF file paths** (one or more PDFs).
2. Loop through every page of every PDF.
3. Extract the text of each page.
4. Keep track of **which file and which page** each piece of text came from — this metadata is
   what lets us show "sources" later when we answer questions.
5. Handle errors gracefully: a corrupted PDF or an empty scanned page shouldn't crash the whole
   pipeline.

> **Note on scanned PDFs:** `pypdf` extracts text that is already stored as text in the PDF. It
> cannot read text that only exists as a picture (a scanned page with no OCR layer). That is a
> known limitation — see Section 15 (Future Improvements).


In [6]:

def load_pdfs(pdf_paths: List[str]) -> List[Document]:
    '''
    Reads one or more PDF files and returns a list of LangChain `Document` objects,
    one per page, each carrying metadata about its source file and page number.

    Args:
        pdf_paths: list of file paths to PDF files on disk.

    Returns:
        List[Document]: each Document has `.page_content` (the extracted text) and
                         `.metadata` (dict with 'source' filename and 'page' number).
    '''
    all_documents = []

    for path in pdf_paths:
        # --- Error handling: skip files that don't exist instead of crashing ---
        if not os.path.exists(path):
            print(f"⚠️  Skipping '{path}' — file not found.")
            continue

        try:
            reader = PdfReader(path)
        except Exception as e:
            # A corrupted or password-protected PDF will raise here.
            print(f"⚠️  Could not open '{path}': {e}")
            continue

        file_name = os.path.basename(path)
        pages_extracted = 0

        for page_number, page in enumerate(reader.pages, start=1):
            try:
                text = page.extract_text() or ""  # extract_text() can return None for blank pages
            except Exception as e:
                print(f"⚠️  Failed to extract page {page_number} of '{file_name}': {e}")
                continue

            # Skip pages with no usable text (e.g. a blank page, or a scanned image page)
            if text.strip() == "":
                continue

            # Wrap the page text in a LangChain Document with useful metadata.
            doc = Document(
                page_content=text,
                metadata={"source": file_name, "page": page_number},
            )
            all_documents.append(doc)
            pages_extracted += 1

        print(f"✅ Loaded '{file_name}': {pages_extracted} page(s) with extractable text.")

    if not all_documents:
        print("❌ No text could be extracted from any of the provided PDFs.")

    return all_documents


# --- Example usage ---
# Place your PDF file(s) in the same folder as this notebook (or give a full path),
# then list them here. You can add as many PDFs as you like.
PDF_PATHS = [
    "sample.pdf",   # <-- replace this with the path to your own PDF
]

documents = load_pdfs(PDF_PATHS)
print(f"\\nTotal pages loaded across all PDFs: {len(documents)}")


✅ Loaded 'sample.pdf': 57 page(s) with extractable text.
\nTotal pages loaded across all PDFs: 57



## 6. Chunking (Splitting Text into Smaller Pieces)

### Why do we need to split text into chunks?

We can't feed an entire PDF into the embedding model or the LLM at once — embedding models work
best on smaller, semantically focused pieces of text, and LLM context windows have limits too.

We use LangChain's **`RecursiveCharacterTextSplitter`**, which:

1. Tries to split on natural boundaries first (paragraphs `\n\n`, then sentences, then words),
   only falling back to splitting mid-word as a last resort — this keeps chunks readable.
2. Uses a **chunk size** (max characters per chunk) and a **chunk overlap** (characters shared
   between consecutive chunks), so that a sentence spanning a chunk boundary doesn't lose context.

### Key parameters

- `chunk_size = 1000` → each chunk is roughly 1000 characters (~150-200 words).
- `chunk_overlap = 150` → each chunk shares its last 150 characters with the next chunk, so
  relevant context isn't lost right at the boundary.

These numbers are a reasonable default for QA tasks; you can tune them based on your documents.


In [7]:

def split_documents(documents: List[Document], chunk_size: int = 1000, chunk_overlap: int = 150) -> List[Document]:
    '''
    Splits a list of page-level Documents into smaller overlapping chunks.

    Args:
        documents: list of Document objects (typically one per PDF page).
        chunk_size: max number of characters per chunk.
        chunk_overlap: number of overlapping characters between consecutive chunks.

    Returns:
        List[Document]: a new, longer list of smaller Document chunks, with metadata preserved.
    '''
    if not documents:
        raise ValueError("No documents provided to split — did Section 5 load any PDFs successfully?")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # These separators are tried in order — paragraph breaks first, then lines, then words.
        separators=["\\n\\n", "\\n", ". ", " ", ""],
    )

    try:
        chunks = splitter.split_documents(documents)
    except Exception as e:
        print("❌ Error while splitting documents:", e)
        raise

    return chunks


# --- Run the chunking step ---
try:
    chunks = split_documents(documents, chunk_size=1000, chunk_overlap=150)
    print(f"✅ Split {len(documents)} page(s) into {len(chunks)} chunk(s).")
    # Show one example chunk so we can sanity-check the output
    if chunks:
        print("\\n--- Example chunk ---")
        print("Source:", chunks[0].metadata)
        print("Text preview:", chunks[0].page_content[:300], "...")
except Exception as e:
    print("❌ Chunking failed:", e)


✅ Split 57 page(s) into 129 chunk(s).
\n--- Example chunk ---
Source: {'source': 'sample.pdf', 'page': 1}
Text preview: Computer  Networks  &Data  Communication  
Notes(22CS008)
 
Data  Communication  
Transferring  data  over  a  transmission  medium  between  two  or  more  devices,  systems,  or  places  is  known  as  data  
communication.
 
Nowadays,
 
computing
 
and
 
telecommunications
 
depend
 
heavily
 
on ...



## 7. Embeddings

### Theory: what is an embedding?

An **embedding** is a way of converting a piece of text into a list of numbers (a *vector*) such
that texts with similar *meaning* end up with similar vectors — even if they use completely
different words.

For example, "the cat sat on the mat" and "a feline rested on the rug" would produce embedding
vectors that are close together in vector space, because a good embedding model captures meaning,
not just exact word overlap.

This is what makes **semantic search** possible: instead of keyword-matching, we can find chunks
that are *conceptually* relevant to a question.

### Model used: `all-MiniLM-L6-v2`

We use the `sentence-transformers/all-MiniLM-L6-v2` model via the `langchain-huggingface`
package's `HuggingFaceEmbeddings` wrapper. It:

- Runs **locally** on your machine (no API key or internet call needed for embeddings).
- Produces 384-dimensional vectors.
- Is small and fast, while still giving strong semantic search quality — a good fit for a
  student project that needs to run without heavy GPU resources.


In [8]:

def get_embedding_model(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    '''
    Loads a HuggingFace sentence-embedding model wrapped for use with LangChain.

    Args:
        model_name: the HuggingFace model identifier.

    Returns:
        HuggingFaceEmbeddings: an embeddings object with `.embed_query()` and `.embed_documents()`.
    '''
    try:
        embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            # Force CPU so this works reliably on any machine, including ones without a GPU.
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},  # normalized vectors -> cleaner similarity scores
        )
        return embeddings
    except Exception as e:
        print("❌ Failed to load embedding model:", e)
        raise


# --- Load the embedding model (this may take a few seconds the first time, as it downloads weights) ---
embedding_model = get_embedding_model()
print("✅ Embedding model loaded.")

# --- Quick sanity check: embed a test sentence and inspect the vector ---
test_vector = embedding_model.embed_query("What is Retrieval Augmented Generation?")
print(f"Example embedding vector length: {len(test_vector)}")
print(f"First 5 values: {test_vector[:5]}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded.
Example embedding vector length: 384
First 5 values: [-0.1085294708609581, -0.027145929634571075, -0.05479291081428528, 0.05426321178674698, -0.023731790482997894]



## 8. Create the FAISS Vector Database

### Theory: what is FAISS and why do we need it?

We now have hundreds (or thousands) of text chunks, each represented as a vector. When a user asks
a question, we need to quickly find the chunks whose vectors are **most similar** to the question's
vector.

Doing this with a naive loop (compare the query vector to every single chunk vector one by one)
becomes slow as the number of chunks grows. **FAISS** (Facebook AI Similarity Search) is a library
built specifically for extremely fast similarity search over large collections of vectors, using
optimized indexing structures.

We use LangChain's `FAISS` vector store wrapper, which:

1. Takes our list of chunks (Documents).
2. Embeds each chunk using the embedding model from Section 7.
3. Builds a searchable FAISS index in memory.
4. Lets us later save/reload this index to disk so we don't have to re-embed every time.


In [9]:

def build_vector_store(chunks: List[Document], embeddings, save_path: str = "faiss_index") -> FAISS:
    '''
    Embeds all chunks and builds a FAISS vector store from them, then saves it to disk.

    Args:
        chunks: list of chunked Document objects.
        embeddings: the embedding model to use (from Section 7).
        save_path: local folder to persist the FAISS index for reuse later.

    Returns:
        FAISS: a searchable vector store.
    '''
    if not chunks:
        raise ValueError("No chunks available to embed — did Section 6 run successfully?")

    try:
        vector_store = FAISS.from_documents(chunks, embeddings)
    except Exception as e:
        print("❌ Failed to build the FAISS vector store:", e)
        raise

    # Persist the index to disk so it can be reloaded later without recomputing embeddings.
    try:
        vector_store.save_local(save_path)
        print(f"✅ FAISS index built and saved to '{save_path}/'.")
    except Exception as e:
        print(f"⚠️  Index built in memory, but saving to disk failed: {e}")

    return vector_store


# --- Build the vector database from our chunks ---
vector_store = build_vector_store(chunks, embedding_model, save_path="faiss_index")
print(f"Total vectors stored in FAISS index: {vector_store.index.ntotal}")


✅ FAISS index built and saved to 'faiss_index/'.
Total vectors stored in FAISS index: 129



## 9. Retriever

### Theory: turning the vector store into a "retriever"

A **retriever** is a simple wrapper around the vector store that, given a query string, returns
the `k` most similar chunks. LangChain exposes this via `.as_retriever(search_kwargs={"k": ...})`.

We choose **k = 4** by default: enough context for the LLM to answer most questions, without
overwhelming the prompt with irrelevant text (which can actually *hurt* answer quality).

We'll also write a small helper function, `retrieve_chunks()`, that we can call directly to inspect
*which* chunks were retrieved for a given question — this is useful both for debugging and for the
"show source chunks" requirement of our project.


In [10]:

def get_retriever(vector_store: FAISS, k: int = 4):
    '''
    Wraps a FAISS vector store as a LangChain retriever.

    Args:
        vector_store: the FAISS vector store built in Section 8.
        k: number of top-matching chunks to retrieve per query.

    Returns:
        A LangChain retriever object.
    '''
    return vector_store.as_retriever(search_type="similarity", search_kwargs={"k": k})


def retrieve_chunks(retriever, query: str) -> List[Document]:
    '''
    Runs a similarity search for the given query and returns the matching chunks.
    Includes basic error handling in case the retriever fails (e.g. empty index).
    '''
    try:
        results = retriever.invoke(query)
        return results
    except Exception as e:
        print("❌ Retrieval failed:", e)
        return []


# --- Build the retriever ---
retriever = get_retriever(vector_store, k=4)

# --- Quick test: retrieve chunks for a sample query ---
sample_query = "What is this document about?"
retrieved = retrieve_chunks(retriever, sample_query)

print(f"Retrieved {len(retrieved)} chunk(s) for query: '{sample_query}'\\n")
for i, doc in enumerate(retrieved, start=1):
    print(f"--- Chunk {i} | Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')} ---")
    print(doc.page_content[:200], "...\\n")


Retrieved 4 chunk(s) for query: 'What is this document about?'\n
--- Chunk 1 | Source: sample.pdf | Page: 33 ---
. ●  Uses  hardware  addresses  to  identify  devices  within  the  same  network  segment. ●  Determines  how  multiple  devices  share  the  same  physical  medium,  avoiding  collisions. 
Working   ...\n
--- Chunk 2 | Source: sample.pdf | Page: 55 ---
.  These  will  be  discussed  in  subsequent  chapters.  ●  Digital  data  to  Digital  signals  −  These  are  in  this  section.  There  are  several  ways  to  map  digital  data  to  digital  sig ...\n
--- Chunk 3 | Source: sample.pdf | Page: 47 ---
.  
●  It  can  route  known  protocols  (like  TCP/IP).  ●  It  bridge  unknown  protocols  (like  non-routable  legacy  traffic)  at  the  same  time  and  rarely  used  today. ...\n
--- Chunk 4 | Source: sample.pdf | Page: 1 ---
Computer  Networks  &Data  Communication  
Notes(22CS008)
 
Data  Communication  
Transferring  data  over  a  transmission  medium  between  two


## 10. Gemini LLM

### Theory: connecting to Google's Gemini API

Now that we can retrieve relevant chunks, we need a language model to turn "question + retrieved
chunks" into a fluent, accurate answer. We use **Google Gemini** via the `langchain-google-genai`
package, which exposes Gemini through the same `ChatModel` interface LangChain uses for every LLM
provider — this is what makes LangChain "provider-agnostic."

### Model choice: `gemini-1.5-flash`

We use `gemini-1.5-flash` because it is fast and cost-efficient, which is ideal for an interactive
Q&A chat interface where users expect quick responses. (You can swap in `gemini-1.5-pro` for higher
quality at the cost of speed, by changing one variable.)

### Key parameter: `temperature`

`temperature` controls how "creative" vs. "deterministic" the model's output is:
- **Low temperature (0–0.3)** → more focused, factual, repeatable answers. Ideal for QA over
  documents, where we want grounded, consistent answers rather than creative variation.
- **High temperature (0.7–1.0)** → more varied, creative output. Better for brainstorming or
  creative writing, not for factual retrieval-based QA.

We use `temperature=0.2` here since accuracy matters more than creativity for this project.


In [11]:

def get_gemini_llm(model_name: str = "gemini-1.5-flash", temperature: float = 0.2):
    '''
    Initializes a Gemini chat model through LangChain.

    Args:
        model_name: which Gemini model to use.
        temperature: 0 = deterministic/factual, 1 = creative/random.

    Returns:
        ChatGoogleGenerativeAI: a LangChain-compatible chat model.
    '''
    try:
        llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=temperature,
            google_api_key=GEMINI_API_KEY,  # loaded in Section 4
        )
        return llm
    except Exception as e:
        print("❌ Failed to initialize the Gemini LLM:", e)
        raise


# --- Initialize the Gemini model ---
llm = get_gemini_llm(
    model_name="gemini-flash-latest",
    temperature=0.2
)

# --- Quick sanity check: send a trivial prompt to confirm the API key and connection work ---
try:
    test_response = llm.invoke("Reply with exactly one word: 'ready'.")
    print("✅ Gemini responded successfully:", test_response.content)
except Exception as e:
    print("❌ Gemini API test call failed. Check your API key and internet connection.")
    print("Error details:", e)


✅ Gemini responded successfully: [{'type': 'text', 'text': 'ready', 'extras': {'signature': 'EvoDCvcDARFNMg/TvSrsmBE0V1UEZPoj2aliGn/VjbBX3fvYF5t9vr19doBvVzYm+IubzwiWA/xe3QWqmphmgJ8imR9WVcNHnqWjxMNaWbjJHjANTmNugNzl+r0+CGilMJMI4hHH1I8NJDUTZqykzJ7Rp3UosLkZ4IeTLW2ZWUiq/7iPkuADXHZEjbv+RgaVN8SirPkgT94RnPFpncXfcpDjVG1CUsfwp7iMPsP0xV9EClFF63FfAICg5kEEyMZS3tRMbxnkTbifxYFpJItHzmp6WnCmQ22BOe+Tv5zoV+fkJEp6ga/a6Zm/+fm4Hnn9sBGJYJGDtOQP+RzTZVh7QHtfx01EueDqHgCmuPPRwJ/LXH6KX/EVXewlDUqTKjlzK9rz9hf32llceOZFsMm9+DfLVkwvdi9bRzOnNWQ5H7D3cVK8DbNLRPdbHgwTxxs6561gSSWmEwPBta5I90sHYoccIT/NtbIzm8reqCyl5HPSQS7aFus4Ffjl7DFLYRWYGrjF+7b50poTD/fETiXBLI64HSDDOlDo+HPRwSseIbVg/rvIX2wBPp2Gf6T44oNkUG+2OV30Tfmkhqh91VAX+fkxCPWFDTci8fZosdzJFbY5IvqWWgoVm/knpFIXcUW+ZComN4U81ZfmSqm71J+6nqZvieoP0g3ZZuQ+NDs='}}]



## 11. RAG Pipeline (Putting It All Together)

### Theory: designing the prompt

This is the most important design decision in a RAG system: **how do we instruct the LLM to use
the retrieved chunks correctly?**

Our prompt template does three things:

1. **Provides the retrieved context** — the top-k chunks from Section 9, clearly labeled with
   their source file and page number.
2. **Provides the user's question.**
3. **Gives explicit instructions**: *answer only using the provided context; if the answer isn't
   in the context, say so instead of guessing.* This is what keeps the system "grounded" and
   reduces hallucination — the single biggest advantage of RAG over asking a raw LLM.

### The pipeline function

`answer_question()` combines every piece built so far:

1. Retrieve top-k relevant chunks for the question.
2. Format them into a context block.
3. Build the final prompt.
4. Call Gemini.
5. Return both the **answer** and the **source chunks used** (so the UI can display them).


In [12]:
# The prompt template that grounds Gemini's answer in the retrieved context.
# {context} and {question} are filled in dynamically for every question asked.
RAG_PROMPT_TEMPLATE = """You are a helpful assistant answering questions about a set of PDF documents.

Use ONLY the following retrieved context to answer the question.

If the answer is not contained in the context, say:
"I could not find this information in the provided document(s)."

Do not make up information that isn't in the context.

Context:
{context}

Question:
{question}

Answer (be concise and accurate):
"""


def format_context(chunks: List[Document]) -> str:
    """
    Formats retrieved chunks into a readable context block for Gemini.
    Each chunk is labeled with its source filename and page number.
    """
    formatted_parts = []

    for i, doc in enumerate(chunks, start=1):
        source = doc.metadata.get("source", "Unknown File")
        page = doc.metadata.get("page", "?")

        formatted_parts.append(
            f"[Chunk {i} | Source: {source} | Page: {page}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_parts)


def answer_question(question: str, retriever, llm) -> Tuple[str, List[Document]]:
    """
    Complete RAG pipeline.

    Steps:
    1. Retrieve relevant chunks.
    2. Build prompt.
    3. Send prompt to Gemini.
    4. Return answer and retrieved chunks.
    """

    if not question.strip():
        return "Please enter a valid question.", []

    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_chunks(retriever, question)

    if not retrieved_chunks:
        return "No relevant information was found in the uploaded document(s).", []

    # Step 2: Build context
    context_text = format_context(retrieved_chunks)

    # Step 3: Create prompt
    prompt = RAG_PROMPT_TEMPLATE.format(
        context=context_text,
        question=question
    )

    # Step 4: Generate answer using Gemini
    try:
        response = llm.invoke(prompt)

        # Gemini may return either a string or a structured list
        if isinstance(response.content, str):
            answer = response.content

        elif isinstance(response.content, list):
            text_parts = []

            for item in response.content:
                if isinstance(item, dict):
                    if item.get("type") == "text":
                        text_parts.append(item.get("text", ""))

                elif hasattr(item, "text"):
                    text_parts.append(item.text)

                else:
                    text_parts.append(str(item))

            answer = "\n".join(text_parts)

        else:
            answer = str(response.content)

    except Exception as e:
        print("❌ Error while generating an answer from Gemini:")
        print(e)
        return f"Error while generating answer:\n{e}", retrieved_chunks

    return answer, retrieved_chunks


# ----------------------------------------------------------
# Test the complete RAG pipeline
# ----------------------------------------------------------

test_question = "Summarize the main topic of this document in two sentences."

answer, sources = answer_question(
    test_question,
    retriever,
    llm
)

print("Question:")
print(test_question)

print("\nAnswer:")
print(answer)

print(f"\nRetrieved {len(sources)} source chunk(s):")

for doc in sources:
    print(
        f"• Source: {doc.metadata.get('source')} | "
        f"Page: {doc.metadata.get('page')}"
    )

Question:
Summarize the main topic of this document in two sentences.

Answer:
This document covers computer networks and data communication, which involves transferring data over a transmission medium between devices or systems. It details components of communication systems alongside networking concepts like TCP/IP, network performance management, and mechanisms for regulating data transmission using frames and addresses.

Retrieved 4 source chunk(s):
• Source: sample.pdf | Page: 33
• Source: sample.pdf | Page: 45
• Source: sample.pdf | Page: 1
• Source: sample.pdf | Page: 39



## 12. Chat History

### Theory: why maintain chat history?

Right now, `answer_question()` treats every question in isolation. But a real chat assistant needs
**session memory** so a user can ask natural follow-ups like *"What about page 3?"* or *"Can you
explain that in simpler terms?"* without repeating the full question.

For this project, we implement a lightweight, transparent approach to chat history rather than a
complex memory framework, so the mechanics stay clear for a beginner:

1. We keep a simple Python list, `chat_history`, where each entry is
   `(question, answer, source_chunks)`.
2. Every time a new question is asked, we display the full running conversation.
3. (Optional extension) We could feed the last N turns back into the prompt so Gemini has
   conversational context — we implement a simple version of this below via
   `build_conversational_question()`, which prepends recent history to the new question before
   retrieval, so follow-up questions still retrieve relevant chunks correctly.


In [13]:
# Global in-memory chat history
chat_history = []


def build_conversational_question(question: str, history, max_turns: int = 3):
    """
    Build a context-aware question using recent chat history.
    """

    if len(history) == 0:
        return question

    recent_history = history[-max_turns:]

    conversation = ""

    for turn in recent_history:
        conversation += f"Q: {turn['question']}\n"
        conversation += f"A: {turn['answer']}\n\n"

    return (
        "Previous conversation:\n"
        f"{conversation}"
        f"Current question: {question}"
    )


def ask_with_history(question: str, retriever, llm, history):
    """
    Ask a question while keeping track of conversation history.
    """

    enriched_question = build_conversational_question(
        question,
        history
    )

    answer, sources = answer_question(
        enriched_question,
        retriever,
        llm
    )

    # Just in case the answer is not a string
    if not isinstance(answer, str):
        answer = str(answer)

    turn = {
        "question": question,
        "answer": answer,
        "sources": sources
    }

    history.append(turn)

    return turn


# -------------------------------------------------------
# Test Conversation
# -------------------------------------------------------

chat_history.clear()

turn1 = ask_with_history(
    "What is this document about?",
    retriever,
    llm,
    chat_history
)

print("Q1:", turn1["question"])
print("A1:", turn1["answer"])


turn2 = ask_with_history(
    "Can you give more detail on that?",
    retriever,
    llm,
    chat_history
)

print("\nQ2:", turn2["question"])
print("A2:", turn2["answer"])

print(f"\nChat history contains {len(chat_history)} turns.")

Q1: What is this document about?
A1: Based on the provided context, the document contains notes on **Computer Networks & Data Communication** (Notes 22CS008). It covers topics such as data communication definitions, system components, transmission channels/types, digital data and signals, and networking concepts/protocols like TCP/IP.

Q2: Can you give more detail on that?
A2: Based on the provided context, here are additional details on the topics covered in the document:

* **Data Communication Definition & Purpose:** It is the transfer of data over a transmission medium between two or more devices, systems, or places, enabling applications like email, video chatting, and the Internet.
* **System Components:** One key component is the **Message**, which is the information being transmitted (such as text, audio, or video files).
* **TCP (Transmission Control Protocol):** 
  * Divides messages into packets at the source computer and reassembles them at the destination.
  * Ensures pack

In [14]:
turn3 = ask_with_history(
    "Explain that in simpler language.",
    retriever,
    llm,
    chat_history
)

print("Q3:", turn3["question"])
print("A3:", turn3["answer"])

Q3: Explain that in simpler language.
A3: Here is a simpler explanation of the main ideas from the document:

* **Data Communication:** Moving information (like text, audio, or video files) from one computer or device to another over cables or wireless signals so apps like email and video chat can work.
* **Message:** The actual piece of information or file being sent.
* **TCP (Transmission Control Protocol):** A set of rules that chops large messages into smaller pieces (segments/packets), attaches sending and receiving addresses, numbers them so they stay in order, and rebuilds them properly at the destination without mistakes.

**How sending data works in simple steps:**
1. **Creation:** An app (like a web browser or email client) creates the data.
2. **Breakdown:** The Transport Layer splits the data into smaller segments and assigns port numbers so it reaches the right app.
3. **Addressing:** The Internet Layer puts IP addresses on the segments so they can navigate across networks


## 13. Gradio Interface

### Theory: why Gradio inside a notebook?

`Gradio` lets us build a web-based chat UI with a few lines of Python, and — unlike Streamlit — it
can launch **directly inside a Jupyter/Colab notebook cell** (`inline=True`), which is exactly what
this project requires.

Our interface needs to support the full project requirement list:

1. **Upload one or more PDFs** — `gr.File(file_count="multiple")`.
2. **Process button** — runs Sections 5–9 (load → chunk → embed → index → retriever) on the
   newly uploaded files.
3. **Chatbot window** — displays the conversation using `gr.Chatbot`.
4. **Textbox + Send button** — for typing questions.
5. **Source chunks display** — a text panel showing exactly which chunks were used for the most
   recent answer, satisfying the "display source chunks" requirement transparently.

We wire all of this together with a few callback functions and `gr.Blocks()`, which gives us full
layout control (rows/columns) compared to the simpler `gr.Interface`.


In [15]:
# ---- Global state holders for the Gradio app ----
# These get replaced whenever the user uploads new PDFs and clicks "Process".
app_state = {
    "vector_store": None,
    "retriever": None,
    "chat_history": [],
}


def process_uploaded_pdfs(files):
    '''
    Callback for the 'Process PDFs' button.
    Runs the full ingestion pipeline (Sections 5-9) on newly uploaded files.

    Args:
        files: list of file objects from gr.File (each has a `.name` attribute = temp file path).

    Returns:
        str: a status message shown to the user.
    '''
    if not files:
        return "⚠️ Please upload at least one PDF file before processing."

    try:
        # gr.File gives us temporary file paths; we reuse our Section 5 function directly.
        file_paths = [f.name for f in files]
        docs = load_pdfs(file_paths)

        if not docs:
            return "❌ No extractable text was found in the uploaded PDF(s)."

        doc_chunks = split_documents(docs)
        vs = build_vector_store(
            doc_chunks,
            embedding_model,
            save_path="faiss_index"
        )
        rt = get_retriever(vs, k=4)

        # Save into shared app state so the chat callback can use them.
        app_state["vector_store"] = vs
        app_state["retriever"] = rt
        app_state["chat_history"] = []   # reset conversation whenever new documents are loaded

        return (
            f"✅ Processed {len(files)} file(s): "
            f"{len(docs)} page(s) → "
            f"{len(doc_chunks)} chunk(s) indexed. "
            f"You can now ask questions below."
        )

    except Exception as e:
        traceback.print_exc()
        return f"❌ An error occurred while processing PDFs: {e}"


def chat_callback(user_message, gradio_history):
    '''
    Callback triggered when the user sends a message in the chat box.

    Args:
        user_message: the new question typed by the user.
        gradio_history: history maintained by gr.Chatbot.

    Returns:
        (updated_gradio_history, cleared_textbox_value, sources_text)
    '''

    if gradio_history is None:
        gradio_history = []

    retriever_ready = app_state.get("retriever") is not None

    if not retriever_ready:
        gradio_history.append(
            {
                "role": "assistant",
                "content": "⚠️ Please upload and process a PDF first."
            }
        )
        return gradio_history, "", ""

    if not user_message or not user_message.strip():
        return gradio_history, "", ""

    # Run the RAG + chat-history pipeline built in Sections 11-12
    turn = ask_with_history(
        user_message,
        app_state["retriever"],
        llm,
        app_state["chat_history"]
    )

    # Build a readable "sources" panel showing exactly which chunks were used
    if turn["sources"]:
        source_lines = [
            f"• {doc.metadata.get('source')} (page {doc.metadata.get('page')})"
            for doc in turn["sources"]
        ]
        sources_text = (
            "Sources used for this answer:\n"
            + "\n".join(source_lines)
        )
    else:
        sources_text = "No source chunks were retrieved for this answer."

    # Add messages in Gradio 6 format
    gradio_history.append(
        {
            "role": "user",
            "content": user_message
        }
    )

    gradio_history.append(
        {
            "role": "assistant",
            "content": turn["answer"]
        }
    )

    return gradio_history, "", sources_text


def clear_chat():
    '''
    Resets the visible chat window and the underlying chat history state.
    '''
    app_state["chat_history"] = []
    return [], ""


# ---- Build the Gradio Blocks layout ----
with gr.Blocks(title="PDF Question Answering (RAG + Gemini)") as demo:

    gr.Markdown("# 📄 AI-Powered PDF Question Answering System (RAG + Gemini)")

    gr.Markdown(
        "Upload one or more PDFs, click **Process PDFs**, then ask questions about their content."
    )

    with gr.Row():

        pdf_upload = gr.File(
            label="Upload PDF file(s)",
            file_count="multiple",
            file_types=[".pdf"]
        )

        process_btn = gr.Button(
            "Process PDFs",
            variant="primary"
        )

    status_box = gr.Textbox(
        label="Status",
        interactive=False
    )

    with gr.Row():

        with gr.Column(scale=2):

            chatbot = gr.Chatbot(
                label="Chat",
                height=400,
                
            )

            question_box = gr.Textbox(
                label="Ask a question about your PDF(s)",
                placeholder="Type your question and press Enter..."
            )

            with gr.Row():
                send_btn = gr.Button(
                    "Send",
                    variant="primary"
                )

                clear_btn = gr.Button(
                    "Clear Chat"
                )

        with gr.Column(scale=1):

            sources_box = gr.Textbox(
                label="Source chunks used",
                lines=15,
                interactive=False
            )

    # ---- Wire up the callbacks to the UI components ----
    process_btn.click(
        fn=process_uploaded_pdfs,
        inputs=[pdf_upload],
        outputs=[status_box]
    )

    send_btn.click(
        fn=chat_callback,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box, sources_box]
    )

    question_box.submit(
        fn=chat_callback,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box, sources_box]
    )

    clear_btn.click(
        fn=clear_chat,
        inputs=[],
        outputs=[chatbot, sources_box]
    )

print("✅ Gradio interface defined. Run the next cell to launch it.")

✅ Gradio interface defined. Run the next cell to launch it.


In [16]:

# Launch the Gradio app.
# `inline=True` embeds the UI directly inside this notebook's output cell.
# `share=False` keeps it local; set share=True to get a temporary public link (e.g. for a demo).
# We wrap the launch in a try/except so port conflicts give a clear message instead of a crash.
try:
    demo.launch(inline=True, share=False, debug=False)
except Exception as e:
    print("❌ Failed to launch Gradio app:", e)
    print("Tip: if a port is already in use, try demo.launch(server_port=7861) instead.")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


✅ Loaded 'EJ1172284.pdf': 11 page(s) with extractable text.
✅ FAISS index built and saved to 'faiss_index/'.



## 14. Evaluation Examples

### Theory: how do we know if our RAG system is actually good?

It's not enough for the system to "run without errors" — for a course project, we should
demonstrate that it produces **correct, grounded, and traceable** answers. We evaluate along three
simple dimensions that don't require any extra libraries:

1. **Groundedness** — does the answer only use information present in the retrieved chunks (no
   hallucinated facts)?
2. **Relevance of retrieval** — are the retrieved chunks actually related to the question?
3. **Refusal correctness** — when we ask something the document does *not* cover, does the system
   correctly say "I don't know" rather than making something up?

Below we run a small evaluation set of questions and manually inspect the results. In a larger
project, you could extend this with automatic metrics (e.g. ROUGE/BLEU against reference answers,
or an "LLM-as-judge" scoring approach), but manual inspection is appropriate and expected for a
course project of this scope.


In [17]:

def run_evaluation(questions: List[str], retriever, llm) -> None:
    '''
    Runs a list of evaluation questions through the RAG pipeline and prints a
    structured report: question, answer, and the sources used — so correctness
    and groundedness can be manually verified.
    '''
    print("=" * 80)
    print("EVALUATION REPORT")
    print("=" * 80)

    for i, q in enumerate(questions, start=1):
        print(f"\\n[{i}] Question: {q}")
        try:
            answer, sources = answer_question(q, retriever, llm)
            print(f"    Answer: {answer}")
            if sources:
                print("    Sources used:")
                for doc in sources:
                    print(f"      - {doc.metadata.get('source')} (page {doc.metadata.get('page')})")
            else:
                print("    Sources used: none (no relevant chunks retrieved)")
        except Exception as e:
            print(f"    ❌ Evaluation failed for this question: {e}")

    print("\\n" + "=" * 80)
    print("END OF EVALUATION REPORT")
    print("=" * 80)


# --- Define a small evaluation set covering different question types ---
evaluation_questions = [
    "What is the main topic of this document?",                     # broad, should be answerable
    "Can you summarize the key points in 3 bullet points?",          # summarization ability
    "What does the document say about [some specific term]?",       # replace with a real term from YOUR pdf
    "What is the capital of France?",                                # NOT in the document -> should refuse
]

# --- Run the evaluation (only if a PDF has already been processed/indexed) ---
if app_state.get("retriever") is not None:
    run_evaluation(evaluation_questions, app_state["retriever"], llm)
else:
    # Fall back to the retriever built earlier in the notebook (Section 9) if the
    # Gradio UI hasn't been used to upload a new file yet.
    run_evaluation(evaluation_questions, retriever, llm)


EVALUATION REPORT
\n[1] Question: What is the main topic of this document?
    Answer: Based on the provided context, the main topic is a study examining students' experiences and behaviors regarding the use of mobile devices for learning the English language.
    Sources used:
      - EJ1172284.pdf (page 11)
      - EJ1172284.pdf (page 5)
      - EJ1172284.pdf (page 4)
      - EJ1172284.pdf (page 9)
\n[2] Question: Can you summarize the key points in 3 bullet points?
    Answer: Based on the provided document context, here are the key points summarized in three bullet points:

* **Concept of Learner Autonomy:** Learner autonomy is described as a multidimensional, multi-faceted concept involving a learner's capacity for self-directed learning, critical reflection, independent action, and taking control of their own learning.
* **Methodology and Data Analysis:** The study collected data via interviews conducted in Polish (translated by the author) and analyzed the data both qualitativel


## 15. Future Improvements

This notebook implements a complete, working RAG pipeline end-to-end, but there are several
meaningful directions to extend it further — useful both as a roadmap for extending this project
and as talking points if you present or discuss this project (e.g. in a viva or interview):

1. **OCR support for scanned PDFs** — integrate `pytesseract` or a vision-language model to handle
   PDFs that are scanned images rather than text layers, which `pypdf` cannot read.

2. **Better chunking strategies** — use semantic chunking (splitting based on topic/meaning shifts
   detected via embeddings) instead of fixed character counts, for more coherent chunks.

3. **Re-ranking retrieved chunks** — add a cross-encoder re-ranker (e.g. from
   `sentence-transformers`) after the initial FAISS retrieval to improve precision on the final
   top-k chunks passed to the LLM.

4. **Hybrid search** — combine dense vector search (what we built) with traditional keyword search
   (BM25) to catch exact-match terms (e.g. product codes, names) that embeddings sometimes miss.

5. **Persistent, multi-session chat memory** — store chat history in a database (e.g. SQLite) so
   conversations persist across notebook restarts, rather than living only in the `chat_history`
   Python list.

6. **Streaming responses** — use Gemini's streaming API so answers appear token-by-token in the
   Gradio UI instead of all at once, improving perceived responsiveness.

7. **Agentic extensions** — turn this RAG system into a true agent (per this course's CLO05) by
   giving the LLM tools it can decide to call: a calculator, a web-search tool for information
   outside the PDF, or an "email this summary" action — using a framework like LangGraph or
   CrewAI, as introduced later in the course syllabus.

8. **Automatic evaluation metrics** — replace the manual evaluation in Section 14 with automated
   scoring, e.g. an "LLM-as-judge" approach where a second Gemini call rates each answer's
   faithfulness to the retrieved context on a numeric scale.

9. **Multi-modal support** — extend retrieval to also index images/diagrams/tables extracted from
   PDFs (using a vision-capable Gemini model), not just body text.

10. **Deployment** — package this notebook's logic into a standalone Python app (FastAPI backend +
    Gradio or React frontend) so it can be deployed as a real web service rather than run inside a
    notebook.

---

### 🎓 Summary

You have now built a complete, working, end-to-end **Retrieval-Augmented Generation** system:
PDF ingestion → chunking → embeddings → FAISS vector search → grounded LLM answering with Gemini →
chat memory → an interactive Gradio interface → and a basic evaluation harness. This mirrors the
architecture of production RAG systems used in real-world applications like enterprise document
search, customer support bots, and research assistants.


In [36]:
import gradio as gr

print(gr.__version__)
print(gr.__file__)

6.22.0
C:\Users\manik\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\__init__.py


In [38]:
import inspect
import gradio as gr

print(inspect.signature(gr.Chatbot))

(value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_classes: 'list[str] | str | None' = None, autoscroll: 'bool' = True, render: 'bool' = True, key: 'int | str | tuple[int | str, ...] | None' = None, preserved_by_key: 'list[str] | str | None' = 'value', height: 'int | str | None' = 400, resizable: 'bool' = False, max_height: 'int | str | None' = None, min_height: 'int | str | None' = None, editable: "Literal['user', 'all'] | None" = None, latex_delimiters: 'list[dict[str, str | bool]] | None' = None, rtl: 'bool' = False, buttons: "list[Literal['share', 'copy', 'copy_all'] | Button] | None" = None, watermark: 'str | None' = None, avatar

In [21]:
from google import genai
import getpass

# Secure prompt in Jupyter/Python to enter your API key
api_key = getpass.getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=api_key)

for model in client.models.list():
    print(model.name)

Enter your Gemini API Key:  ········


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-